# Esplorazione iniziale — BTC & ETH (Fase 1)

Primo notebook di EDA. Carica i dati salvati in `data/raw/yahoo/crypto/` e
produce statistiche descrittive, distribuzione dei rendimenti, autocorrelazione.

**Prerequisito**: aver eseguito `uv run python -m src.ingestion.tier1.fetch_tier1`
in un ambiente con accesso alla rete (vedi `docs/data_sources_tier1.md`).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.assets.asset import TIER1_ASSETS

DATA_DIR = Path("../data/raw/yahoo/crypto")
sns.set_theme(style="whitegrid")

In [ ]:
def load_ohlcv(symbol: str) -> pd.DataFrame:
    path = DATA_DIR / f"{symbol}_1d.parquet"
    df = pd.read_parquet(path)
    df["log_return"] = np.log(df["close"]).diff()
    return df

btc = load_ohlcv("BTC")
eth = load_ohlcv("ETH")
print(f"BTC: {len(btc)} rows, {btc.index.min().date()} → {btc.index.max().date()}")
print(f"ETH: {len(eth)} rows, {eth.index.min().date()} → {eth.index.max().date()}")

## Statistica descrittiva dei rendimenti

In [ ]:
summary = pd.DataFrame({
    "BTC": btc["log_return"].describe(),
    "ETH": eth["log_return"].describe(),
})
summary.loc["skew"] = [btc["log_return"].skew(), eth["log_return"].skew()]
summary.loc["kurtosis"] = [btc["log_return"].kurtosis(), eth["log_return"].kurtosis()]
summary.loc["annualized_vol"] = summary.loc["std"] * np.sqrt(365)
summary

## Distribuzione dei rendimenti — quanto sono lontani dalla normale?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, df, name in zip(axes, [btc, eth], ["BTC", "ETH"]):
    sns.histplot(df["log_return"].dropna(), bins=80, kde=True, ax=ax)
    ax.set_title(f"{name} daily log returns")
    ax.set_xlabel("log return")
plt.tight_layout()

## Autocorrelazione dei rendimenti e della |rendimento|

Se i rendimenti hanno autocorrelazione zero ma |rendimenti| no, abbiamo
il classico **volatility clustering**: i mercati sono efficienti in media
ma la varianza è prevedibile.

In [ ]:
from pandas.plotting import autocorrelation_plot

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for col, df, name in zip([0, 1], [btc, eth], ["BTC", "ETH"]):
    r = df["log_return"].dropna()
    axes[0, col].stem(range(1, 31), [r.autocorr(lag=k) for k in range(1, 31)])
    axes[0, col].set_title(f"{name} ACF(log_return)")
    axes[0, col].axhline(0, color="k", lw=0.5)
    axes[1, col].stem(range(1, 31), [r.abs().autocorr(lag=k) for k in range(1, 31)])
    axes[1, col].set_title(f"{name} ACF(|log_return|)")
    axes[1, col].axhline(0, color="k", lw=0.5)
plt.tight_layout()

## Note per iterazioni successive

- Aggiungere SOL, LINK, POL una volta confermato che il fetch li copre
- Aggiungere context assets (SPX, NDX, DXY, GOLD) e calcolare correlazioni
  rolling con BTC/ETH
- Identificare visivamente i regimi (bull/bear/sideways) e marcare gli
  eventi noti (FTX collapse, ETF approval, halving)
- Stagionalità: rendimenti per giorno della settimana, mese
- Cluster di volatilità: visualizzare la persistenza